In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./data/restaurants.csv')

In [3]:
df.head()

,id,Restaurant name,cusine,price_range,location,dietary_tags,features
0,1,The Plaza,Nigerian Staples,Premium,Mainland,Halal,"Buffet, Event Hall, Jollof Rice, Family-friendly"
1,2,Black Bell,Nigerian Staples,Moderate,Mainland,Halal,"Everyday Eats, Jollof Rice, Delivery, Casual"
2,3,Aunty Fati's Food,Nigerian Staples,Budget,Mainland,Halal,"Swallow, Affordable, Traditional, Quick bite"
3,4,Exodus,Breakfast,Moderate,Island,Vegetarian,"Pancakes, Coffee, Brunch, Outdoor seating"
4,5,Fast Chicken,Fast Food,Budget,Mainland,Halal,"Fried Chicken, Quick bite, Late-night, Takeout"


In [4]:
df['dietary_tags'] = df['dietary_tags'].apply(lambda x: [t.strip() for t in str(x).split(',')] if pd.notna(x) and str(x) != 'nan' else [])
df['features'] = df['features'].apply(lambda x: [f.strip() for f in str(x).split(',')] if pd.notna(x) and str(x) != 'nan' else [])

df

,id,Restaurant name,cusine,price_range,location,dietary_tags,features
0,1,The Plaza,Nigerian Staples,Premium,Mainland,[Halal],"[Buffet, Event Hall, Jollof Rice, Family-frien..."
1,2,Black Bell,Nigerian Staples,Moderate,Mainland,[Halal],"[Everyday Eats, Jollof Rice, Delivery, Casual]"
2,3,Aunty Fati's Food,Nigerian Staples,Budget,Mainland,[Halal],"[Swallow, Affordable, Traditional, Quick bite]"
3,4,Exodus,Breakfast,Moderate,Island,[Vegetarian],"[Pancakes, Coffee, Brunch, Outdoor seating]"
4,5,Fast Chicken,Fast Food,Budget,Mainland,[Halal],"[Fried Chicken, Quick bite, Late-night, Takeout]"
5,6,Tasty Turkey,Fast food,Moderate,Island,[Halal],"[Grilled Turkey, Fries, Casual, Delivery]"
6,7,Crimp's Pizza,Fast food,Premium,Island,"[Halal, Nut-free, Vegetarian]","[Gourmet Pizza, Cocktails, Modern, Delivery]"
7,8,Uncle P's Pastries,Pastry,Moderate,Mainland,[Halal],"[Snacks, Cakes, Coffee, Quick bite]"
8,9,Alaba Oge,Traditional,Budget,Mainland,[Halal],"[Amala, Swallow, Spicy, Street Food]"
9,10,Naija Buka Express,Traditional,Budget,Mainland,[Halal],"[Mild, Buka Classics, Swallow, Quick bite]"


In [5]:
df.rename(columns={'Restaurant name': 'restaurant_name'}, inplace=True)
df.rename(columns={'cusine': 'cuisine'}, inplace=True)

In [6]:
def create_text_representation(row):
    features_str = ", ".join(row['features'])
    dietary_str = ", ".join(row['dietary_tags']) or "Standard"
    return f"{row['restaurant_name']} is a {row['price_range']} {row['cuisine']} restaurant located in {row['location']}. Key features: {features_str}. Dietary tags: {dietary_str}."

df['text_representation'] = df.apply(create_text_representation, axis=1)

In [7]:
df['text_representation'].head()

0    The Plaza is a Premium Nigerian Staples restau...
1    Black Bell is a Moderate Nigerian Staples rest...
2    Aunty Fati's Food is a Budget Nigerian Staples...
3    Exodus is a Moderate Breakfast restaurant loca...
4    Fast Chicken is a Budget Fast Food restaurant ...
Name: text_representation, dtype: str

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
restaurant_embeddings = model.encode(df['text_representation'].tolist())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
restaurant_embeddings
print("Embedding Shape:", restaurant_embeddings.shape)

Embedding Shape: (25, 768)


In [10]:
def build_user_preferences(raw_query, cuisine=None, price_range=None, location=None, dietary_restrictions=None, features=None):
    dietary_restrictions = dietary_restrictions or []
    features = features or []
    
    hard_constraints = ["dietary_restrictions"] if dietary_restrictions else []
    
    soft_targets = [("cuisine", cuisine), ("price_range", price_range), ("location", location), ("features", features)]
    soft_preferences = [name for name, val in soft_targets if val]
        
    return {
        "raw_query": raw_query,
        "cuisine": cuisine,
        "price_range": price_range,
        "location": location,
        "dietary_restrictions": dietary_restrictions,
        "features": features,
        "hard_constraints": hard_constraints,
        "soft_preferences": soft_preferences
    }

In [11]:
prefs_full = build_user_preferences(
    raw_query="I need cheap vegetarian breakfast on the mainland.",
    cuisine="breakfast",
    price_range="Budget",
    location="mainland",
    dietary_restrictions=["Vegetarian"]
)

prefs_partial = build_user_preferences(
    raw_query="I want asian food.",
    cuisine="asian"
)

print(prefs_full)
print(prefs_partial)

{'raw_query': 'I need cheap vegetarian breakfast on the mainland.', 'cuisine': 'breakfast', 'price_range': 'Budget', 'location': 'mainland', 'dietary_restrictions': ['Vegetarian'], 'features': [], 'hard_constraints': ['dietary_restrictions'], 'soft_preferences': ['cuisine', 'price_range', 'location']}
{'raw_query': 'I want asian food.', 'cuisine': 'asian', 'price_range': None, 'location': None, 'dietary_restrictions': [], 'features': [], 'hard_constraints': [], 'soft_preferences': ['cuisine']}


In [12]:
test_queries = [
    "Budget traditional Nigerian food on the Mainland",
    "Vegetarian breakfast options on the Island",
    "Premium seafood or fine dining on the Island",
    "Spicy late-night street food and finger foods"
]

In [13]:
PRICE_MAP = {
    "budget": "Budget",
    "cheap": "Budget",
    "affordable": "Budget",
    "moderate": "Moderate",
    "premium": "Premium",
    "expensive": "Premium"
}

LOCATION_MAP = {
    "mainland": "Mainland",
    "island": "Island"
}

CUISINE_MAP = {
    "nigerian": "Nigerian Staples",
    "breakfast": "Breakfast",
    "fast food": "Fast Food",
    "pastry": "Pastry",
    "pastries": "Pastry",
    "traditional": "Traditional",
    "seafood": "Seafood",
    "intercontinental": "Intercontinental",
    "continental": "Intercontinental",
    "asian fusion": "Asian Fusion",
    "asian": "Asian Fusion",
    "finger foods": "Finger Foods",
    "dessert": "Dessert"
}

DIETARY_MAP = {
    "vegetarian": "Vegetarian",
    "halal": "Halal",
    "pescatarian": "Pescatarian",
    "nut-free": "Nut-Free",
    "gluten-free": "Gluten-Free",
    "dairy-free": "Dairy-Free"
}

In [14]:
def get_first_match(mapping, text):
    for kw, val in mapping.items():
        if kw in text:
            return val
    return None

def extract_preferences(raw_query):
    query_lower = raw_query.lower()
    
    return build_user_preferences(
        raw_query=raw_query,
        cuisine=get_first_match(CUISINE_MAP, query_lower),
        price_range=get_first_match(PRICE_MAP, query_lower),
        location=get_first_match(LOCATION_MAP, query_lower),
        dietary_restrictions=[val for kw, val in DIETARY_MAP.items() if kw in query_lower]
    )

In [15]:
test_queries = [
    "Budget traditional Nigerian food on the Mainland",
    "Vegetarian breakfast options on the Island",
    "Premium seafood or fine dining on the Island",
    "Spicy street food "
]

for q in test_queries:
    parsed = extract_preferences(q)
    print(f"Query: {parsed['raw_query']}")
    print(f"Extracted: cuisine={parsed['cuisine']}, price={parsed['price_range']}, location={parsed['location']}, dietary={parsed['dietary_restrictions']}")
    print(f"Hard Constraints: {parsed['hard_constraints']}")
    print(f"Soft Preferences: {parsed['soft_preferences']}")
    print("-" * 60)

Query: Budget traditional Nigerian food on the Mainland
Extracted: cuisine=Nigerian Staples, price=Budget, location=Mainland, dietary=[]
Hard Constraints: []
Soft Preferences: ['cuisine', 'price_range', 'location']
------------------------------------------------------------
Query: Vegetarian breakfast options on the Island
Extracted: cuisine=Breakfast, price=None, location=Island, dietary=['Vegetarian']
Hard Constraints: ['dietary_restrictions']
Soft Preferences: ['cuisine', 'location']
------------------------------------------------------------
Query: Premium seafood or fine dining on the Island
Extracted: cuisine=Seafood, price=Premium, location=Island, dietary=[]
Hard Constraints: []
Soft Preferences: ['cuisine', 'price_range', 'location']
------------------------------------------------------------
Query: Spicy street food 
Extracted: cuisine=None, price=None, location=None, dietary=[]
Hard Constraints: []
Soft Preferences: []
-----------------------------------------------------

In [16]:
def generate_user_embedding(user_prefs, model):
    return model.encode(user_prefs["raw_query"])

In [17]:
sample_prefs = extract_preferences("Spicy street food ")
user_vector = generate_user_embedding(sample_prefs, model)

print("Query:", sample_prefs["raw_query"])
print("Embedding Shape:", user_vector.shape)

Query: Spicy street food 
Embedding Shape: (768,)


In [18]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def calculate_semantic_similarity(user_vector, restaurant_embeddings):
    user_vector_2d = user_vector.reshape(1, -1)
    similarities = cosine_similarity(user_vector_2d, restaurant_embeddings)
    return similarities[0]

In [19]:
sample_prefs = extract_preferences("Spicy street food ")
user_vector = generate_user_embedding(sample_prefs, model)

semantic_scores = calculate_semantic_similarity(user_vector, restaurant_embeddings)

df['semantic_score'] = semantic_scores
print(df[['restaurant_name', 'cuisine', 'location', 'semantic_score']].sort_values(by='semantic_score', ascending=False).head())

          restaurant_name           cuisine  location  semantic_score
14           Spice Bistro  Intercontinental  Mainland        0.516555
22  Gourmet Wraps & Bites      Finger Foods    Island        0.460065
5            Tasty Turkey         Fast food    Island        0.438144
18    Urban Express Diner         Fast food    Island        0.433070
12          The Suya Spot         Fast Food    Island        0.408431


In [20]:
sample_prefs = extract_preferences("Jollof rice ")
user_vector = generate_user_embedding(sample_prefs, model)

semantic_scores = calculate_semantic_similarity(user_vector, restaurant_embeddings)

df['semantic_score'] = semantic_scores
print(df[['restaurant_name', 'cuisine', 'location', 'semantic_score']].sort_values(by='semantic_score', ascending=False).head())

        restaurant_name           cuisine  location  semantic_score
0             The Plaza  Nigerian Staples  Mainland        0.356155
11  Calabar Kitchenette       Traditional  Mainland        0.257629
17       Tokyo to Lagos      Asian Fusion    Island        0.246938
19     Mama Put Central  Nigerian Staples  Mainland        0.243173
2     Aunty Fati's Food  Nigerian Staples  Mainland        0.239859


In [21]:
def exact_match(preference, value):
    if not preference:
        return 0.0
    return 1.0 if str(preference).lower() == str(value).lower() else 0.0

def calculate_structured_scores(df, preferences):
    cuisine_col = 'cuisine' if 'cuisine' in df.columns else 'cusine'
    
    c_scores = [exact_match(preferences['cuisine'], val) for val in df[cuisine_col]]
    l_scores = [exact_match(preferences['location'], val) for val in df['location']]
    p_scores = [exact_match(preferences['price_range'], val) for val in df['price_range']]
    
    return c_scores, l_scores, p_scores

In [22]:
sample_order_history = {
    "user_1": [1, 2, 20],      
    "user_2": [15, 22, 18],    
    "user_3": [],             
}

In [23]:
def infer_preferences_from_history(past_order_restaurant_ids, df):
    if not past_order_restaurant_ids:
        return {"cuisine": None, "price_range": None, "location": None}

    ordered = df[df['id'].isin(past_order_restaurant_ids)]

    if ordered.empty:
        return {"cuisine": None, "price_range": None, "location": None}

    return {
        "cuisine": ordered['cuisine'].mode()[0] if not ordered['cuisine'].mode().empty else None,
        "price_range": ordered['price_range'].mode()[0] if not ordered['price_range'].mode().empty else None,
        "location": ordered['location'].mode()[0] if not ordered['location'].mode().empty else None,
    }

In [24]:
print(infer_preferences_from_history(sample_order_history["user_1"], df))
print(infer_preferences_from_history(sample_order_history["user_2"], df))
print(infer_preferences_from_history(sample_order_history["user_3"], df))

{'cuisine': 'Nigerian Staples', 'price_range': 'Budget', 'location': 'Mainland'}
{'cuisine': 'Asian Fusion', 'price_range': 'Premium', 'location': 'Island'}
{'cuisine': None, 'price_range': None, 'location': None}


In [25]:
sample_prefs = extract_preferences("Budget traditional Nigerian food on the Mainland")

c_scores, l_scores, p_scores = calculate_structured_scores(df, sample_prefs)

df['cuisine_score'] = c_scores
df['location_score'] = l_scores
df['price_score'] = p_scores

print(df[['restaurant_name', 'cuisine', 'location', 'price_range', 'cuisine_score', 'location_score', 'price_score']].head())

     restaurant_name           cuisine  location price_range  cuisine_score  \
0          The Plaza  Nigerian Staples  Mainland     Premium            1.0   
1         Black Bell  Nigerian Staples  Mainland    Moderate            1.0   
2  Aunty Fati's Food  Nigerian Staples  Mainland      Budget            1.0   
3             Exodus         Breakfast    Island    Moderate            0.0   
4       Fast Chicken         Fast Food  Mainland      Budget            0.0   

   location_score  price_score  
0             1.0          0.0  
1             1.0          0.0  
2             1.0          1.0  
3             0.0          0.0  
4             1.0          1.0  


In [27]:
def normalize_tags(x):
    if isinstance(x, (list, tuple)):
        return [str(tag).strip().lower() for tag in x]
    if isinstance(x, str):
        return [tag.strip().lower() for tag in x.split(',') if tag.strip()]
    return []

df['dietary_tags'] = df['dietary_tags'].apply(normalize_tags)

In [28]:
def apply_hard_constraints(df, preferences):
    dietary_reqs = [req.lower() for req in preferences.get("dietary_restrictions", [])]
    if not dietary_reqs:
        return df
        
    mask = df['dietary_tags'].apply(lambda tags: all(req in tags for req in dietary_reqs))
    return df[mask]

In [29]:
def calculate_final_score(df, preferences, base_weights=None):
    weights = base_weights or {
        'semantic_score_norm': 0.4,
        'cuisine_score_norm': 0.2,
        'location_score_norm': 0.2,
        'price_score_norm': 0.2,
    }
    
    mapping = {
        'cuisine': 'cuisine_score_norm',
        'location': 'location_score_norm',
        'price_range': 'price_score_norm'
    }
    
    active_weights = {'semantic_score_norm': weights['semantic_score_norm']}
    for key, col in mapping.items():
        if preferences.get(key):
            active_weights[col] = weights[col]
            
    total_weight = sum(active_weights.values())
    
    df_scored = df.copy()
    df_scored['final_score'] = sum(
        df_scored[col] * (weight / total_weight)
        for col, weight in active_weights.items()
        if col in df_scored.columns
    )
    return df_scored.sort_values(by='final_score', ascending=False)

In [30]:
def get_top_k_recommendations(df, k=3):
    top_k = df.head(k)
    results = []
    
    cuisine_col = 'cuisine' if 'cuisine' in top_k.columns else 'cusine'
    
    for rank, (_, row) in enumerate(top_k.iterrows(), 1):
        rec = {
            'Rank': rank,
            'Restaurant Name': row['restaurant_name'],
            'Cuisine': row[cuisine_col],
            'Location': row['location'],
            'Price': row['price_range'],
            'Match Score': f"{row['final_score'] * 100:.1f}%"
        }
        results.append(rec)
        
    return pd.DataFrame(results)

In [31]:
def recommend(raw_query, df, model, restaurant_embeddings, k=3, weights=None, past_order_restaurant_ids=None):
    prefs = extract_preferences(raw_query)

    if past_order_restaurant_ids:
        inferred = infer_preferences_from_history(past_order_restaurant_ids, df)
        for key in ['cuisine', 'price_range', 'location']:
            if prefs[key] is None:
                prefs[key] = inferred.get(key)

    scored = df.copy()

    user_vector = generate_user_embedding(prefs, model)
    scored['semantic_score'] = calculate_semantic_similarity(user_vector, restaurant_embeddings)

    smin, smax = scored['semantic_score'].min(), scored['semantic_score'].max()
    denom = smax - smin if smax != smin else 1.0
    scored['semantic_score_norm'] = (scored['semantic_score'] - smin) / denom

    c_scores, l_scores, p_scores = calculate_structured_scores(scored, prefs)
    scored['cuisine_score_norm'] = c_scores
    scored['location_score_norm'] = l_scores
    scored['price_score_norm'] = p_scores

    filtered = apply_hard_constraints(scored, prefs)
    final = calculate_final_score(filtered, prefs, weights)
    
    return get_top_k_recommendations(final, k=k)

In [32]:
top_recommendations = recommend("Vegetarian breakfast options on the Island", df, model, restaurant_embeddings, k=3)
print(top_recommendations.to_string(index=False)) 

 Rank          Restaurant Name   Cuisine Location    Price Match Score
    1 Sunrise Pastries & Diner Breakfast   Island   Budget      100.0%
    2                   Exodus Breakfast   Island Moderate       88.1%
    3      The Morning Griddle Breakfast Mainland Moderate       64.7%


In [33]:
top_recommendations = recommend("pastries", df, model, restaurant_embeddings, k=3)
print(top_recommendations.to_string(index=False)) 

 Rank          Restaurant Name   Cuisine Location    Price Match Score
    1       Uncle P's Pastries    Pastry Mainland Moderate      100.0%
    2 Sunrise Pastries & Diner Breakfast   Island   Budget       65.4%
    3       Cold Treats Corner   Dessert   Island  Premium       35.7%


In [34]:
top_recommendations = recommend("Asian food", df, model, restaurant_embeddings, k=3)
print(top_recommendations.to_string(index=False)) 

 Rank   Restaurant Name      Cuisine Location    Price Match Score
    1    Tokyo to Lagos Asian Fusion   Island  Premium      100.0%
    2        Wok & Roll Asian Fusion   Island  Premium       90.5%
    3 123 Chops & Grill Finger Foods Mainland Moderate       57.4%


In [35]:
top_recommendations = recommend("Suya", df, model, restaurant_embeddings, k=3)
print(top_recommendations.to_string(index=False)) 

 Rank    Restaurant Name      Cuisine Location   Price Match Score
    1      The Suya Spot    Fast Food   Island  Budget      100.0%
    2 Naija Buka Express  Traditional Mainland  Budget       42.8%
    3     Tokyo to Lagos Asian Fusion   Island Premium       40.8%


In [36]:
top_recommendations = recommend("Cheap food mainland", df, model, restaurant_embeddings, k=3)
print(top_recommendations.to_string(index=False)) 

 Rank   Restaurant Name          Cuisine Location  Price Match Score
    1      Fast Chicken        Fast Food Mainland Budget      100.0%
    2 Aunty Fati's Food Nigerian Staples Mainland Budget       88.6%
    3         Alaba Oge      Traditional Mainland Budget       80.4%


In [37]:
top_recommendations = recommend("continental", df, model, restaurant_embeddings, k=3)
print(top_recommendations.to_string(index=False)) 

 Rank     Restaurant Name          Cuisine Location    Price Match Score
    1         Vintage Pot Intercontinental   Island  Premium       77.4%
    2 Urban Express Diner        Fast food   Island Moderate       66.7%
    3      Coastal Grills          Seafood   Island  Premium       59.0%


In [38]:
print("Query only, no history")
print(recommend("something good", df, model, restaurant_embeddings, k=3).to_string(index=False))

print("\n No specific query, using user_1's order history")
print(recommend("food", df, model, restaurant_embeddings, k=3,
                 past_order_restaurant_ids=sample_order_history["user_1"]).to_string(index=False))

print("\n Explicit query overrides history (user_1 asking for Premium)")
print(recommend("Premium seafood", df, model, restaurant_embeddings, k=3,
                 past_order_restaurant_ids=sample_order_history["user_1"]).to_string(index=False))


print("\n New user (user_3), no history, cold start")
print(recommend("Asian food", df, model, restaurant_embeddings, k=3,
                 past_order_restaurant_ids=sample_order_history["user_3"]).to_string(index=False))

Query only, no history
 Rank       Restaurant Name      Cuisine Location    Price Match Score
    1          Tasty Turkey    Fast food   Island Moderate      100.0%
    2 Gourmet Wraps & Bites Finger Foods   Island Moderate       87.0%
    3    Cold Treats Corner      Dessert   Island  Premium       86.1%

 No specific query, using user_1's order history
 Rank   Restaurant Name          Cuisine Location   Price Match Score
    1 Aunty Fati's Food Nigerian Staples Mainland  Budget       97.6%
    2  Mama Put Central Nigerian Staples Mainland  Budget       84.9%
    3         The Plaza Nigerian Staples Mainland Premium       61.9%

 Explicit query overrides history (user_1 asking for Premium)
 Rank Restaurant Name          Cuisine Location   Price Match Score
    1  Coastal Grills          Seafood   Island Premium       80.0%
    2       The Plaza Nigerian Staples Mainland Premium       55.0%
    3  Tokyo to Lagos     Asian Fusion   Island Premium       36.1%

 New user (user_3), no hist